# Interactive Visual Analytics with Folium

The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
!pip install folium
!pip install pandas
!pip install plotly

In [3]:
import folium
import pandas as pd
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [4]:
import folium

# Define launch site coordinates
launch_sites = {
    "CCAFS LC-40": (28.562302, -80.577356),
    "KSC LC-39A": (28.573255, -80.646895),
    "VAFB SLC-4E": (34.632093, -120.610829),
    "Boca Chica": (25.9972, -97.1566)
}

# Create a base map centered roughly on the US
m = folium.Map(location=[30, -95], zoom_start=4)

# Add markers for each launch site
for site, coords in launch_sites.items():
    folium.Marker(
        location=coords,
        popup=site,
        tooltip=site,
        icon=folium.Icon(color="red", icon="rocket", prefix="fa")
    ).add_to(m)

# Display the map
m

First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
import pandas as pd

URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"

# Pandas can read directly from the URL
spacex_df = pd.read_csv(URL)

# Preview the first 5 rows
spacex_df.head(5)

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

# Initial map centered on NASA JSC
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle and a Marker
for i, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # Circle with popup
    circle = folium.Circle(
        location=coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # Text label marker
    marker = folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site_name
        )
    )
    
    # Add to map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


##- Equator proximity
- Florida sites (Cape Canaveral, Kennedy Space Center) → ~28°N latitude, closer to the equator than most U.S. territory.
- Boca Chica (Texas) → ~26°N latitude, even closer to the equator.
- Vandenberg (California) → ~34°N latitude, farther north, not near the equator.
 Not all sites are near the equator — only Florida and Texas are relatively close.
- Coastal proximity
- All sites are right on the coast: Florida (Atlantic Ocean), Texas (Gulf of Mexico), California (Pacific Ocean).
 Yes, all launch sites are very close to coastlines.
- Why this matters
- Equator advantage: Launches closer to the equator get a “boost” from Earth’s rotation, useful for equatorial and geostationary orbits.
- Coastal advantage: Launching over water "reduces risk to populated areas and allows safe trajectories for different orbital inclinations.
- Strategic diversity: Florida/Texas handle equatorial/geostationary launches, California handles polar/sun-synchronous launches.


In [10]:
# Task 2: Mark the success/failed launches for each site on the map
import folium
from folium.features import DivIcon

# Start map centered on NASA JSC
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Loop through each launch record in spacex_df
for i, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    outcome = row['class']   # 1 = success, 0 = failure
    
    # Choose color based on success/failure
    color = 'green' if outcome == 1 else 'red'
    
    # Circle marker with popup
    circle = folium.Circle(
        location=coordinate,
        radius=1000,
        color=color,
        fill=True
    ).add_child(folium.Popup(f"{site_name} - {'Success' if outcome == 1 else 'Failure'}"))
    
    # Text label marker
    marker = folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12px; color:{color};"><b>{site_name}</b></div>'
        )
    )
    
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

In [11]:
# Print all successful launches
print("✅ Successful Launches:")
print(spacex_df[spacex_df['class'] == 1][['Launch Site', 'Lat', 'Long']])

# Print all failed launches
print("\n❌ Failed Launches:")
print(spacex_df[spacex_df['class'] == 0][['Launch Site', 'Lat', 'Long']])

✅ Successful Launches:
     Launch Site        Lat        Long
17   CCAFS LC-40  28.562302  -80.577356
18   CCAFS LC-40  28.562302  -80.577356
20   CCAFS LC-40  28.562302  -80.577356
21   CCAFS LC-40  28.562302  -80.577356
22   CCAFS LC-40  28.562302  -80.577356
24   CCAFS LC-40  28.562302  -80.577356
25   CCAFS LC-40  28.562302  -80.577356
28   VAFB SLC-4E  34.632834 -120.610745
29   VAFB SLC-4E  34.632834 -120.610745
30   VAFB SLC-4E  34.632834 -120.610745
31   VAFB SLC-4E  34.632834 -120.610745
36    KSC LC-39A  28.573255  -80.646895
38    KSC LC-39A  28.573255  -80.646895
39    KSC LC-39A  28.573255  -80.646895
41    KSC LC-39A  28.573255  -80.646895
42    KSC LC-39A  28.573255  -80.646895
44    KSC LC-39A  28.573255  -80.646895
45    KSC LC-39A  28.573255  -80.646895
46    KSC LC-39A  28.573255  -80.646895
47    KSC LC-39A  28.573255  -80.646895
48    KSC LC-39A  28.573255  -80.646895
49  CCAFS SLC-40  28.563197  -80.576820
50  CCAFS SLC-40  28.563197  -80.576820
54  CCAFS SLC-40 

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [12]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [13]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [14]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map

In [15]:
import folium
from folium.plugins import MarkerCluster

# Create a new column for marker colors based on class
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Initialize the map centered on NASA JSC
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Create a MarkerCluster object
marker_cluster = MarkerCluster()

# For each launch record, add a marker to the cluster
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"{record['Launch Site']} - {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

# Add marker cluster to the map
site_map.add_child(marker_cluster)

# Display the map
site_map

From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [16]:
# TASK 3: Calculate the distances between a launch site to its proximities
import numpy as np

# Haversine function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Example: distance from each launch site to NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]

launch_sites_df['Distance_to_NASA_JSC_km'] = launch_sites_df.apply(
    lambda row: haversine(row['Lat'], row['Long'], nasa_coordinate[0], nasa_coordinate[1]),
    axis=1
)

print(launch_sites_df[['Launch Site', 'Distance_to_NASA_JSC_km']])

    Launch Site  Distance_to_NASA_JSC_km
0   CCAFS LC-40              1413.328592
1  CCAFS SLC-40              1413.366602
2    KSC LC-39A              1406.434271
3   VAFB SLC-4E              2462.753721


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [17]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [18]:
import numpy as np

# Haversine formula to calculate distance in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Example: distance from a launch site to a nearby railway
launch_site = [28.562302, -80.577356]   # CCAFS LC-40
railway_point = [28.5721, -80.5850]     # coordinates you noted from MousePosition

distance_km = haversine(launch_site[0], launch_site[1], railway_point[0], railway_point[1])
print(f"Distance from launch site to railway: {distance_km:.2f} km")


Distance from launch site to railway: 1.32 km


In [19]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [20]:
launch_site = [28.562302, -80.577356]      # CCAFS LC-40
coastline_point = [28.56390, -80.57120]    # closer coastline coordinate

distance_km = haversine(launch_site[0], launch_site[1], coastline_point[0], coastline_point[1])
print(f"Distance: {distance_km:.2f} km")

Distance: 0.63 km


In [21]:
launch_site = [28.562302, -80.577356]      # CCAFS LC-40
coastline_point = [28.56367, -80.57163]    # Closest coastline

distance_km = haversine(launch_site[0], launch_site[1], coastline_point[0], coastline_point[1])
print(f"Distance: {distance_km:.2f} km")

Distance: 0.58 km


In [22]:
launch_site = [28.562302, -80.577356]      # CCAFS LC-40
coastline_point = [28.56395, -80.57110]    # closer coastline coordinate

distance_km = haversine(launch_site[0], launch_site[1],
                        coastline_point[0], coastline_point[1])
print(f"Distance: {distance_km:.2f} km")

Distance: 0.64 km


In [23]:
# Example coastline points you marked
coastline_points = [
    [28.56367, -80.57163],
    [28.56395, -80.57110],
    [28.56420, -80.57080]
]

launch_site = [28.562302, -80.577356]  # CCAFS LC-40

# Calculate all distances
distances = [haversine(launch_site[0], launch_site[1], lat, lon) for lat, lon in coastline_points]

# Find the closest
min_distance = min(distances)
closest_point = coastline_points[distances.index(min_distance)]

print(f"Closest coastline point: {closest_point}, Distance: {min_distance:.2f} km")

Closest coastline point: [28.56367, -80.57163], Distance: 0.58 km


In [24]:
# Candidate coastline points (from MousePosition)
coastline_points = [
    [28.56367, -80.57163],
    [28.56395, -80.57110],
    [28.56420, -80.57080]
]

launch_site = [28.562302, -80.577356]  # CCAFS LC-40

# Calculate all distances
distances = [haversine(launch_site[0], launch_site[1], lat, lon) for lat, lon in coastline_points]

# Find the closest
min_distance = min(distances)
closest_point = coastline_points[distances.index(min_distance)]

print(f"Closest coastline point: {closest_point}, Distance: {min_distance:.2f} km")

Closest coastline point: [28.56367, -80.57163], Distance: 0.58 km


In [25]:
# Candidate coastline points (from MousePosition)
coastline_points = [
    [28.56367, -80.57163],
    [28.56395, -80.57110],
    [28.56420, -80.57080]
]

launch_site = [28.562302, -80.577356]  # CCAFS LC-40

# Calculate all distances
distances = [haversine(launch_site[0], launch_site[1], lat, lon) for lat, lon in coastline_points]

# Find the closest
min_distance = min(distances)
closest_point = coastline_points[distances.index(min_distance)]

print(f"Closest coastline point: {closest_point}, Distance: {min_distance:.2f} km")

Closest coastline point: [28.56367, -80.57163], Distance: 0.58 km


In [26]:
import folium
from folium.features import DivIcon
import numpy as np

# Haversine formula to calculate distance in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Example: Launch site coordinate (CCAFS LC-40)
launch_site = [28.562302, -80.577356]

# Closest coastline coordinate (from MousePosition plugin)
coastline_point = [28.56367, -80.57163]

# Calculate distance
distance_km = haversine(launch_site[0], launch_site[1], coastline_point[0], coastline_point[1])

# Initialize map
site_map = folium.Map(location=launch_site, zoom_start=13)

# Add marker for coastline point with distance label
distance_marker = folium.Marker(
    location=coastline_point,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_km)
    )
)

# Add both launch site and coastline markers
folium.Marker(location=launch_site, popup="Launch Site").add_to(site_map)
site_map.add_child(distance_marker)

# Display map
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [27]:
import folium
from folium.features import DivIcon
import numpy as np

# Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Example coordinates
launch_site = [28.562302, -80.577356]      # CCAFS LC-40
coastline_point = [28.56367, -80.57163]    # Closest coastline (from MousePosition)

# Calculate distance
distance_km = haversine(launch_site[0], launch_site[1], coastline_point[0], coastline_point[1])

# Initialize map
site_map = folium.Map(location=launch_site, zoom_start=13)

# Add markers
folium.Marker(location=launch_site, popup="Launch Site").add_to(site_map)
folium.Marker(location=coastline_point, popup="Coastline").add_to(site_map)

# Add distance label at coastline point
distance_marker = folium.Marker(
    location=coastline_point,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_km)
    )
)
site_map.add_child(distance_marker)

# Draw PolyLine between launch site and coastline
lines = folium.PolyLine(locations=[launch_site, coastline_point], weight=2, color='blue')
site_map.add_child(lines)

# Display map
site_map

In [28]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


In [29]:
import folium
from folium.features import DivIcon
import numpy as np

# Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Example: Launch site (CCAFS LC-40)
launch_site = [28.562302, -80.577356]

# Example proximity coordinates (replace with MousePosition values)
closest_city = [28.538336, -81.379234]     # Orlando
closest_railway = [28.5721, -80.5850]      # Example railway
closest_highway = [28.5645, -80.5700]      # Example highway

# Calculate distances
city_distance = haversine(*launch_site, *closest_city)
railway_distance = haversine(*launch_site, *closest_railway)
highway_distance = haversine(*launch_site, *closest_highway)

# Initialize map
site_map = folium.Map(location=launch_site, zoom_start=10)

# Add launch site marker
folium.Marker(location=launch_site, popup="Launch Site").add_to(site_map)

# Add proximity markers with distance labels
for point, label, dist in [
    (closest_city, "City", city_distance),
    (closest_railway, "Railway", railway_distance),
    (closest_highway, "Highway", highway_distance)
]:
    folium.Marker(location=point, popup=label).add_to(site_map)
    distance_marker = folium.Marker(
        location=point,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(dist)
        )
    )
    site_map.add_child(distance_marker)

    # Draw PolyLine between launch site and proximity
    line = folium.PolyLine(locations=[launch_site, point], weight=2, color='blue')
    site_map.add_child(line)

# Display map
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


- Railways 
Launch sites are generally not right next to railways. Railways tend to be a few kilometers away. This makes sense because heavy rail traffic could pose risks, and launch sites need clear safety zones.
- Highways }
Highways are usually nearby but not too close. They provide access for transporting rockets and equipment, but the sites keep some distance to avoid danger to public traffic during launches.
- Coastlines 
All launch sites are very close to coastlines. This is intentional: launching over the ocean reduces risk to populated areas if something goes wrong, and it provides a clear flight path for rockets.
- Cities 
Launch sites are kept far away from major cities. This is a safety measure — launches involve explosive fuel and possible debris, so sites are located in relatively remote areas to protect people.


- Railways: not immediate neighbors, but accessible.
- Highways: nearby for logistics, but not too close.
- Coastlines: always very close — safety and flight path reasons.
- Cities: kept at a safe distance to protect populations.





This pattern shows that launch sites are carefully chosen to balance safety, accessibility, and practicality.
